In [167]:
import asyncio
import glob
import httpx
import logging
import pandas as pd
import polars as pl
from datetime import datetime, timezone
from dependency_metrics.analyzer import DependencyAnalyzer
from packaging import version as pkg_version
from packaging.requirements import Requirement
from packaging.specifiers import SpecifierSet
from pathlib import Path
from typing import Any, Optional, Union

## Import Deps.Dev Data

In [168]:
path_pattern = "/workspaces/mads-siads-699-winter-2026-capstone/notebooks/data/source_data/initial_dataset/*.parquet"

In [169]:
all_files = sorted(glob.glob(path_pattern))
first_x = all_files[8:10]
if first_x:
    df_deps = pl.read_parquet(first_x, n_rows=200)
    print(f"Loaded {len(first_x)} files.")

Loaded 2 files.


In [170]:
df_deps.head(30)

SnapshotAt,package_name,package_version,package_published_at,dep_name,dep_version,MinimumDepth,github_repo
datetime[μs],str,str,datetime[μs],str,str,i64,str
2023-09-11 21:01:25.316347,"""nwbwidgets""","""0.10.2""",2023-02-24 23:18:41,"""typing-extensions""","""4.7.1""",3,"""neurodatawithoutborders/nwb-ju…"
2023-09-11 21:01:25.316347,"""nwbwidgets""","""0.10.2""",2023-02-24 23:18:41,"""tzdata""","""2023.3.0""",3,"""neurodatawithoutborders/nwb-ju…"
2023-09-11 21:01:25.316347,"""nwbwidgets""","""0.10.2""",2023-02-24 23:18:41,"""uri-template""","""1.3.0""",4,"""neurodatawithoutborders/nwb-ju…"
2023-09-11 21:01:25.316347,"""nwbwidgets""","""0.10.2""",2023-02-24 23:18:41,"""urllib3""","""1.25.11""",2,"""neurodatawithoutborders/nwb-ju…"
2023-09-11 21:01:25.316347,"""nwbwidgets""","""0.10.2""",2023-02-24 23:18:41,"""wcwidth""","""0.2.6""",4,"""neurodatawithoutborders/nwb-ju…"
…,…,…,…,…,…,…,…
2023-09-11 21:01:25.316347,"""nwbwidgets""","""0.11.3""",2023-07-21 15:59:27,"""certifi""","""2023.7.22""",2,"""neurodatawithoutborders/nwb-ju…"
2023-09-11 21:01:25.316347,"""nwbwidgets""","""0.11.3""",2023-07-21 15:59:27,"""cffi""","""1.15.1""",5,"""neurodatawithoutborders/nwb-ju…"
2023-09-11 21:01:25.316347,"""nwbwidgets""","""0.11.3""",2023-07-21 15:59:27,"""charset-normalizer""","""3.2.0""",2,"""neurodatawithoutborders/nwb-ju…"


## Get snapshots of each version
Get the dates (start and end) between each version from the deps.dev table using windowing

In [171]:
date_col = "package_published_at"

snapshots = (
    df_deps.select(["package_name", date_col])
    .unique()
    .sort(["package_name", date_col])
)

snapshots_with_bounds = snapshots.with_columns([
    pl.col(date_col).shift(-1).over("package_name").alias("next_snapshot")
])

snapshots_with_bounds = snapshots_with_bounds.filter(pl.col("next_snapshot").is_not_null())

In [172]:
snapshots_with_bounds.head()

package_name,package_published_at,next_snapshot
str,datetime[μs],datetime[μs]
"""nwbwidgets""",2021-08-20 15:38:34,2023-02-24 23:18:41
"""nwbwidgets""",2023-02-24 23:18:41,2023-07-21 15:59:27


## Test Zahan et. Al method of getting TTU / TTR

In [173]:
all_results = []

for row in snapshots_with_bounds.iter_rows(named=True):
    pkg = row['package_name']
    start = row[date_col]
    end = row['next_snapshot']
    
    print(f"Analyzing {pkg} snapshot window: {start} -> {end}")
    
    try:
        analyzer = DependencyAnalyzer(
            ecosystem="pypi",
            package=pkg,
            start_date=start,
            end_date=end,
            weighting_type="exponential",
            half_life=180,
            output_dir=Path("./output")
        )

        results = analyzer.analyze()
        
        all_results.append({
            'package_name': pkg,
            'snapshot_start': start,
            'snapshot_end': end,
            'avg_ttu': results.get('ttu'),
            'avg_ttr': results.get('ttr')
        })
    except Exception as e:
        print(f"Error processing {pkg} at {start}: {e}")

results_df = pd.DataFrame(all_results)

INFO:dependency_metrics.analyzer:Fetching metadata for nwbwidgets...
INFO:dependency_metrics.analyzer:Analyzing version 0.10.2
INFO:dependency_metrics.analyzer:Found 24 dependencies
INFO:dependency_metrics.analyzer:  Analyzing numpy...
INFO:dependency_metrics.analyzer:Analyzing dependency: numpy
INFO:dependency_metrics.analyzer:  Analyzing matplotlib...
INFO:dependency_metrics.analyzer:Analyzing dependency: matplotlib
INFO:dependency_metrics.analyzer:  Analyzing scikit-image...
INFO:dependency_metrics.analyzer:Analyzing dependency: scikit-image
INFO:dependency_metrics.analyzer:  Analyzing pynwb...
INFO:dependency_metrics.analyzer:Analyzing dependency: pynwb
INFO:dependency_metrics.analyzer:  Analyzing ipympl...
INFO:dependency_metrics.analyzer:Analyzing dependency: ipympl
INFO:dependency_metrics.analyzer:  Analyzing ipydatagrid...
INFO:dependency_metrics.analyzer:Analyzing dependency: ipydatagrid
INFO:dependency_metrics.analyzer:  Analyzing ipyvolume...
INFO:dependency_metrics.analyzer

Analyzing nwbwidgets snapshot window: 2021-08-20 15:38:34 -> 2023-02-24 23:18:41


INFO:dependency_metrics.analyzer:  Analyzing plotly...
INFO:dependency_metrics.analyzer:Analyzing dependency: plotly
INFO:dependency_metrics.analyzer:  Analyzing tqdm...
INFO:dependency_metrics.analyzer:Analyzing dependency: tqdm
INFO:dependency_metrics.analyzer:  Analyzing zarr...
INFO:dependency_metrics.analyzer:Analyzing dependency: zarr
INFO:dependency_metrics.analyzer:  Analyzing tifffile...
INFO:dependency_metrics.analyzer:Analyzing dependency: tifffile
INFO:dependency_metrics.analyzer:  Analyzing ndx-spectrum...
INFO:dependency_metrics.analyzer:Analyzing dependency: ndx-spectrum
INFO:dependency_metrics.analyzer:  Analyzing ndx-icephys-meta...
INFO:dependency_metrics.analyzer:Analyzing dependency: ndx-icephys-meta
INFO:dependency_metrics.analyzer:  Analyzing ndx-grayscalevolume...
INFO:dependency_metrics.analyzer:Analyzing dependency: ndx-grayscalevolume
INFO:dependency_metrics.analyzer:  Analyzing trimesh...
INFO:dependency_metrics.analyzer:Analyzing dependency: trimesh
INFO:dep

Analyzing nwbwidgets snapshot window: 2023-02-24 23:18:41 -> 2023-07-21 15:59:27


INFO:dependency_metrics.analyzer:  Analyzing trimesh...
INFO:dependency_metrics.analyzer:Analyzing dependency: trimesh
INFO:dependency_metrics.analyzer:  Analyzing dandi...
INFO:dependency_metrics.analyzer:Analyzing dependency: dandi
INFO:dependency_metrics.analyzer:  Analyzing importlib-metadata...
INFO:dependency_metrics.analyzer:Analyzing dependency: importlib-metadata
INFO:dependency_metrics.analyzer:  Analyzing fsspec...
INFO:dependency_metrics.analyzer:Analyzing dependency: fsspec
INFO:dependency_metrics.analyzer:  Analyzing requests...
INFO:dependency_metrics.analyzer:Analyzing dependency: requests
INFO:dependency_metrics.analyzer:  Analyzing aiohttp...
INFO:dependency_metrics.analyzer:Analyzing dependency: aiohttp
INFO:dependency_metrics.analyzer:  Analyzing ipydatawidgets...
INFO:dependency_metrics.analyzer:Analyzing dependency: ipydatawidgets
INFO:dependency_metrics.analyzer:  Analyzing ipyfilechooser...
INFO:dependency_metrics.analyzer:Analyzing dependency: ipyfilechooser
IN

In [174]:
results_df.head()

,package_name,snapshot_start,snapshot_end,avg_ttu,avg_ttr
0,nwbwidgets,2021-08-20 15:38:34,2023-02-24 23:18:41,12.098764,0.0
1,nwbwidgets,2023-02-24 23:18:41,2023-07-21 15:59:27,4.958062,0.0


## Try to replicate the Zahan et. Al method of getting TTU/TTR, but in this notebook instead of using dependency-metrics
The dependency-metrics forces you to save all of the metadata locally, which we don't want while we are running this analysis. 

In [175]:
# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
logging.getLogger("httpx").setLevel(logging.WARNING)

# Constants for connection pooling
LIMITS = httpx.Limits(max_connections=100, max_keepalive_connections=20)
CLIENT = httpx.AsyncClient(limits=LIMITS, timeout=10.0)

# Type Aliases
MetadataDict = dict[str, Any]
VulnerabilityList = list[dict[str, Any]]

# In-memory caches
METADATA_CACHE: dict[str, MetadataDict] = {}
VULN_CACHE: dict[str, VulnerabilityList] = {}


def ensure_utc(dt_val: Union[str, datetime, None]) -> Optional[datetime]:
    """
    Ensure a datetime object is UTC aware.

    :param dt_val: The datetime object or ISO string to convert.
    :return: A UTC-aware datetime object or None if input is None.
    """
    if dt_val is None:
        return None
    if isinstance(dt_val, str):
        # Handle 'Z' suffix and parse to datetime
        dt_val = datetime.fromisoformat(dt_val.replace("Z", "+00:00"))
    if dt_val.tzinfo is None:
        return dt_val.replace(tzinfo=timezone.utc)
    return dt_val.astimezone(timezone.utc)


async def fetch_pypi_metadata(package_name: str) -> Optional[MetadataDict]:
    """
    Fetch and cache metadata for a package from PyPI.

    :param package_name: Name of the PyPI package.
    :return: Dictionary of package metadata or None if request fails.
    """
    if package_name in METADATA_CACHE:
        return METADATA_CACHE[package_name]

    url = f"https://pypi.org/pypi/{package_name}/json"
    try:
        response = await CLIENT.get(url)
        if response.status_code == 200:
            data = response.json()
            processed_releases: list[tuple[pkg_version.Version, Optional[datetime], str]] = []

            for ver_str, files in data.get('releases', {}).items():
                if files:
                    try:
                        ver_obj = pkg_version.parse(ver_str)
                        # Use the first file's upload time as release date
                        rel_date = ensure_utc(files[0].get('upload_time'))
                        processed_releases.append((ver_obj, rel_date, ver_str))
                    except (pkg_version.InvalidVersion, ValueError):
                        continue

            # Sort releases by version object for later traversal
            data['processed_releases'] = sorted(processed_releases, key=lambda x: x[0])
            METADATA_CACHE[package_name] = data
            return data
    except httpx.RequestError as exc:
        logger.error("Failed to fetch metadata for %s: %s", package_name, exc)

    return None


async def get_vulnerabilities_osv(package_name: str, version: str) -> VulnerabilityList:
    """
    Query the OSV database for vulnerabilities for a specific package version.

    :param package_name: The name of the package.
    :param version: The specific version string.
    :return: A list of vulnerability objects.
    """
    cache_key = f"{package_name}@{version}"
    if cache_key in VULN_CACHE:
        return VULN_CACHE[cache_key]

    url = "https://api.osv.dev/v1/query"
    query = {
        "version": version,
        "package": {"name": package_name, "ecosystem": "PyPI"}
    }
    try:
        response = await CLIENT.post(url, json=query)
        vulns: VulnerabilityList = response.json().get("vulns", []) if response.status_code == 200 else []
        VULN_CACHE[cache_key] = vulns
        return vulns
    except (httpx.RequestError, ValueError) as exc:
        logger.error("OSV query failed for %s: %s", cache_key, exc)
        return []


def get_best_version_fast(
    metadata: Optional[MetadataDict],
    constraint_str: str,
    before_date: datetime
) -> tuple[Optional[str], Optional[datetime]]:
    """
    Find the newest version matching constraints released before a specific date.

    :param metadata: Processed metadata dictionary.
    :param constraint_str: Version specifier (e.g., ">=1.0").
    :param before_date: Cut-off date for release.
    :return: A tuple of (version_string, release_date).
    """
    if not metadata or 'processed_releases' not in metadata:
        return None, None

    spec = None
    if constraint_str and constraint_str != "*":
        try:
            spec = SpecifierSet(constraint_str)
        except ValueError:
            return None, None

    # Reverse iterate to find the most recent matching version
    for ver_obj, rel_date, ver_str in reversed(metadata['processed_releases']):
        if rel_date and rel_date <= before_date:
            if spec is None or ver_obj in spec:
                return ver_str, rel_date
    return None, None


async def analyze_snapshot_async(row: dict[str, Any]) -> dict[str, Any]:
    """
    Analyze a single snapshot row for TTU and TTR metrics.

    :param row: A dictionary representing a row from the Polars DataFrame.
    :return: Dictionary containing calculated metrics.
    """
    pkg_name = row["package_name"]
    start_date = ensure_utc(row["package_published_at"])
    end_date = ensure_utc(row["next_snapshot"])

    # Fallback return values
    empty_res = {
        "package_name": pkg_name,
        "snapshot_start": row["package_published_at"],
        "snapshot_end": row["next_snapshot"],
        "avg_ttu": 0.0,
        "avg_ttr": 0.0
    }

    if not start_date or not end_date:
        return empty_res

    metadata = await fetch_pypi_metadata(pkg_name)
    if not metadata:
        return empty_res

    parent_ver, _ = get_best_version_fast(metadata, "*", start_date)
    if not parent_ver:
        return empty_res

    try:
        ver_url = f"https://pypi.org/pypi/{pkg_name}/{parent_ver}/json"
        ver_res = await CLIENT.get(ver_url)
        deps_list: list[str] = ver_res.json().get("info", {}).get("requires_dist", []) or []
    except (httpx.RequestError, ValueError):
        return empty_res

    ttu_sums: list[float] = []
    ttr_sums: list[float] = []

    for dep_raw in deps_list:
        try:
            req = Requirement(dep_raw)
            if req.marker and not req.marker.evaluate():
                continue

            dep_meta = await fetch_pypi_metadata(req.name)
            if not dep_meta:
                continue

            curr_v_str, curr_v_date = get_best_version_fast(
                dep_meta, str(req.specifier), start_date
            )
            if not curr_v_str or not curr_v_date:
                continue

            curr_v = pkg_version.parse(curr_v_str)

            # TTU: Time to Update
            outdated_days = 0.0
            for rel_v, rel_date, _ in dep_meta['processed_releases']:
                if rel_date and rel_v > curr_v and rel_date < end_date:
                    lag_start = max(start_date, rel_date)
                    outdated_days = max(
                        outdated_days,
                        (end_date - lag_start).total_seconds() / 86400
                    )
            ttu_sums.append(outdated_days)

            # TTR: Time to Remedy
            vulns = await get_vulnerabilities_osv(req.name, curr_v_str)
            max_ttr = 0.0
            for vuln in vulns:
                for affected in vuln.get("affected", []):
                    if affected.get("package", {}).get("name") != req.name:
                        continue
                    for v_range in affected.get("ranges", []):
                        for event in v_range.get("events", []):
                            if "fixed" in event:
                                _, fix_date = get_best_version_fast(
                                    dep_meta, f"=={event['fixed']}", end_date
                                )
                                if fix_date:
                                    remedy_at = max(start_date, fix_date)
                                    if remedy_at < end_date:
                                        ttr = (end_date - remedy_at).total_seconds() / 86400
                                        max_ttr = max(max_ttr, ttr)
            ttr_sums.append(max_ttr)
        except Exception as exc: # pylint: disable=broad-except
            logger.debug("Error processing dependency %s: %s", dep_raw, exc)
            continue

    return {
        **empty_res,
        "avg_ttu": sum(ttu_sums) / len(ttu_sums) if ttu_sums else 0.0,
        "avg_ttr": sum(ttr_sums) / len(ttr_sums) if ttr_sums else 0.0
    }


async def process_dataframe_async(df: pl.DataFrame) -> pl.DataFrame:
    """
    Main entry point to process a Polars DataFrame of snapshots.

    :param df: Input Polars DataFrame.
    :return: Resulting Polars DataFrame with metrics.
    """
    tasks = [analyze_snapshot_async(row) for row in df.to_dicts()]
    results = await asyncio.gather(*tasks)
    return pl.DataFrame(results)

result_df = asyncio.run(process_dataframe_async(snapshots_with_bounds))

In [176]:
results_df.head()

,package_name,snapshot_start,snapshot_end,avg_ttu,avg_ttr
0,nwbwidgets,2021-08-20 15:38:34,2023-02-24 23:18:41,12.098764,0.0
1,nwbwidgets,2023-02-24 23:18:41,2023-07-21 15:59:27,4.958062,0.0
